In [1]:
from requests import get, post
from sys import exit
from pprint import pprint
import time

In [2]:
HOST = "https://api.chartmetric.com"
TOKEN = "eVCH88EH3jriTh1ps2OpDtXeDMwlPPM7GY9y2J8jFkBY3RuLtx19DIXzC1sn0Ygm"

res = post(f"{HOST}/api/token", json={"refreshtoken": TOKEN})
if res.status_code != 200:
    print(f"ERROR: received a {res.status_code} instead of 200 from /api/token")
    exit(1)

access_token = res.json()["token"]

def Get(endpoint, params=None):
    res = get(
        f"{HOST}{endpoint}",
        headers={"Authorization": f"Bearer {access_token}"},
        params=params
    )
    if res.status_code != 200:
        print("ERROR:", res.status_code, res.text)
        return None
    return res.json()

In [3]:
genres = Get("/api/genre")
genres

{'obj': [{'id': 501120, 'name': 'pop'},
  {'id': 501121, 'name': 'hip-hop/rap'},
  {'id': 501122, 'name': 'rock'},
  {'id': 501123, 'name': 'metal'},
  {'id': 501124, 'name': 'electronic'},
  {'id': 501125, 'name': 'r&b/soul'},
  {'id': 501126, 'name': 'classical'},
  {'id': 501127, 'name': 'jazz'},
  {'id': 501128, 'name': 'blues'},
  {'id': 501129, 'name': 'devotional & spiritual'},
  {'id': 501130, 'name': 'folk'},
  {'id': 501131, 'name': 'country'},
  {'id': 501132, 'name': 'alternative'},
  {'id': 501133, 'name': 'eras'},
  {'id': 501134, 'name': 'ambient'},
  {'id': 501135, 'name': 'psychedelic'},
  {'id': 501136, 'name': 'spoken / comedy'},
  {'id': 501137, 'name': 'vocal'},
  {'id': 501138, 'name': 'singer/songwriter'},
  {'id': 501139, 'name': 'soundtrack'},
  {'id': 501140, 'name': 'instrumental'},
  {'id': 501141, 'name': 'holiday'},
  {'id': 501142, 'name': 'easy listening'},
  {'id': 501143, 'name': 'worldbeat'},
  {'id': 501144, 'name': 'industrial'},
  {'id': 501145, 'n

In [4]:
# CONSTANTS
REACH_MAP = {
    "1": ("Undiscovered",  0,           100_000),
    "2": ("Developing",    100_000,     500_000),
    "3": ("Mid-level",     500_000,   2_000_000),
    "4": ("Mainstream",  2_000_000,  10_000_000),
    "5": ("Superstar",  10_000_000,  50_000_000),
    "6": ("Legendary",  50_000_000, 999_000_000),
}

AGE_KEY_MAP = {
    "13_17": "ages_13_17",
    "18_24": "ages_18_24",
    "25_34": "ages_25_34",
    "35_44": "ages_35_44",
    "45_64": "ages_45_64",
    "65_":   "ages_65_",
}

metric = "sp_monthly_listeners"

In [5]:
# INPUT HELPERS

def prompt_list(prompt, default=None):
    raw = input(f"{prompt} [{', '.join(default) if default else 'skip'}]: ").strip()
    if not raw and default:
        return default
    return [x.strip() for x in raw.split(",") if x.strip()]

def prompt_int(prompt, default):
    raw = input(f"{prompt} [{default}]: ").strip()
    return int(raw) if raw else default

# MOVIE DEMPOGRAPHICS PLACEHOLDER 

def get_movie_demographics(movie_title):
    return {
        "gender": "female",
        "age": ["18_24", "25_34"],
    }

# API CALLS

def get_audience_data(artist_id, audience_country, desired_interests):
    return {
        "stat": Get(f"/api/artist/{artist_id}/social-audience-stats", params={
            "domain": "instagram", "audienceType": "followers", "statsType": "stat"
        }),
        "demographic": Get(f"/api/artist/{artist_id}/social-audience-stats", params={
            "domain": "instagram", "audienceType": "followers", "statsType": "demographic"
        }),
        "country": Get(f"/api/artist/{artist_id}/social-audience-stats", params={
            "domain": "instagram", "audienceType": "followers", "statsType": "country"
        }) if audience_country else None,
        "interest": Get(f"/api/artist/{artist_id}/social-audience-stats", params={
            "domain": "instagram", "audienceType": "followers", "statsType": "interest"
        }) if desired_interests else None,
    }

In [6]:
# DEMOGRAPHIC FILTER

def matches_demographics(artist, gender, min_gender_total, age, min_age_total,
                          audience_country, min_audience_total, desired_interests):
    artist_id   = artist.get("chartmetric_artist_id") or artist.get("id")
    artist_name = artist.get("name", "Unknown")

    all_data = get_audience_data(artist_id, audience_country, desired_interests)

    # Total followers
    stat = all_data["stat"]
    if not stat or not stat.get("obj"):
        print(f"  ERROR: No stat data for {artist_name}")
        return False, {}
    total_followers = stat["obj"][0].get("followers", 0)
    if not total_followers:
        print(f"  ERROR: No follower count for {artist_name}")
        return False, {}
    print(f"  Followers: {total_followers:,}")

    # Gender Check
    demo = all_data["demographic"]
    if not demo or not demo.get("obj"):
        print(f"  ERROR: No demographic data for {artist_name}")
        return False, {}
    obj          = demo["obj"][0]
    gender_total = round(obj.get(gender, 0) * total_followers)
    if gender_total < min_gender_total:
        return False, {"reason": f"{gender} total {gender_total:,} < min {min_gender_total:,}", "total_followers": total_followers}
    print(f"  {gender}: {gender_total:,}")

    # Age Check
    age_totals = {}
    for bucket in age:
        age_key = AGE_KEY_MAP.get(bucket)
        if not age_key:
            print(f"  ERROR: Age bucket '{bucket}' not recognized.")
            return False, {}
        age_totals[bucket] = round(obj.get(age_key, 0) * total_followers)
    if not all(age_totals[b] >= min_age_total for b in age):
        failing = {b: age_totals[b] for b in age if age_totals[b] < min_age_total}
        return False, {"reason": f"age buckets below threshold: {failing}", "total_followers": total_followers}
    for bucket in age:
        print(f"  {bucket}: {age_totals[bucket]:,}")

    # Audience Country Check
    country_totals = {}
    if audience_country:
        country_obj = all_data["country"]
        if not country_obj or not country_obj.get("obj"):
            print(f"  ERROR: No country data for {artist_name}")
            return False, {}
        entries = country_obj["obj"]
        for code in audience_country:
            match = next((e for e in entries if e.get("code2", "").upper() == code.upper()), None)
            country_totals[code] = round(match.get("weights", 0) * total_followers) if match else 0
        if not all(country_totals[c] >= min_audience_total for c in audience_country):
            failing = {c: country_totals[c] for c in audience_country if country_totals[c] < min_audience_total}
            return False, {"reason": f"countries below threshold: {failing}", "total_followers": total_followers}
        for code in audience_country:
            print(f"  {code}: {country_totals[code]:,}")

    # Interests Check
    if desired_interests:
        interest_obj = all_data["interest"]
        if not interest_obj or not interest_obj.get("obj"):
            print(f"  ERROR: No interest data for {artist_name}")
            return False, {}
        interest_names = [e.get("interest_name", "").lower() for e in interest_obj["obj"]]
        matched = [i for i in desired_interests if i.lower() in interest_names]
        print(f"  Interests matched: {matched if matched else 'none'}")

    return True, {
        "total_followers": total_followers,
        "gender_total":    gender_total,
        "age_totals":      age_totals,
        "country_totals":  country_totals if audience_country else {},
    }

In [7]:
# FILTER AND DISPLAY

def filter_and_display(artists_raw, gender, min_gender_total, age, min_age_total,
                        audience_country, min_audience_total, desired_interests,
                        desired_genres, genre_match_mode, desired_moods, mood_match_mode):

    # Genre/mood filter
    def matches_genre_mood(artist):
        genres_raw = artist.get("genres", "")
        if isinstance(genres_raw, str):
            tags = [t.strip().lower() for t in genres_raw.split(",")]
        else:
            tags = []
        mood_fn  = all if mood_match_mode  == "all" else any
        genre_fn = all if genre_match_mode == "all" else any
        mood_match  = mood_fn(m.lower()  in tags for m in desired_moods)  if desired_moods  else True
        genre_match = genre_fn(g.lower() in tags for g in desired_genres) if desired_genres else True
        return mood_match and genre_match

    artists_filtered = [a for a in artists_raw if matches_genre_mood(a)]
    print(f"Genre/mood filter: {len(artists_filtered)} artists remaining.\n")

    # Demographics filter
    matched_artists = []
    for artist in artists_filtered:
        name = artist.get("name", "Unknown")
        print(f"Checking: {name}...")
        is_match, result = matches_demographics(
            artist, gender, min_gender_total, age, min_age_total,
            audience_country, min_audience_total, desired_interests
        )
        if is_match:
            matched_artists.append({**artist, "demographics": result})
            print(f"  ✅ MATCH")
        else:
            if result:
                print(f"  No match — {result.get('reason', 'unknown reason')}")
                print(f"     total_followers: {result.get('total_followers', 'N/A')}")
        time.sleep(0.3)

    # Results
    print(f"\n── {len(matched_artists)} artists matched your criteria ──\n")
    for a in matched_artists:
        d = a["demographics"]
        print(f"🎵 {a['name']} ({(a.get('code2') or '?').upper()})")
        print(f"   Spotify listeners:  {(a.get('spotify_monthly_listeners') or a.get('sp_monthly_listeners') or 0):,}")
        if a.get("career_stage"):
            print(f"   Career stage:       {a.get('career_stage')}")
        if a.get("recent_momentum"):
            print(f"   Momentum:           {a.get('recent_momentum')}")
        if a.get("similarity"):
            print(f"   Similarity score:   {a.get('similarity'):.2f}")
        print(f"   Total IG followers: {d['total_followers']:,}")
        print(f"   {gender.capitalize()} followers:   {d['gender_total']:,}")
        for bucket, total in d["age_totals"].items():
            print(f"   Age {bucket}:         {total:,}")
        for code, total in d["country_totals"].items():
            print(f"   Audience in {code}:   {total:,}")
        print(f"   Genres: {a.get('genres', 'N/A')}")
        print()

In [8]:
# MODE A - SEARCH BASED ON AUDIENCE DEMOGRAPHICS

def run_mode_a():
    print()
    print("── Mode A: Search Based on Audience Demographics ──")

    prefilled = {}
    use_movie = input("\n  Start from a reference movie? (y / n): ").strip().lower()
    if use_movie == "y":
        movie_title = input("  Enter movie title: ").strip()
        print(f"  Fetching audience data for '{movie_title}'...")
        prefilled = get_movie_demographics(movie_title)
        print(f"  Prefilled — gender: {prefilled['gender']}, age: {prefilled['age']}")

    print("\n── Audience Demographics ──")
    gender             = input(f"  Target gender (female / male) [{prefilled.get('gender', 'female')}]: ").strip().lower() or prefilled.get("gender", "female")
    min_gender_total   = prompt_int("  Min absolute followers of that gender", 50_000)
    age                = prompt_list("  Age buckets (13_17, 18_24, 25_34, 35_44, 45_64, 65_)", prefilled.get("age", ["18_24", "25_34"]))
    min_age_total      = prompt_int("  Min absolute followers per age bucket", 10_000)
    audience_country   = prompt_list("  Audience countries (e.g. AR, PE, US)", [])
    min_audience_total = prompt_int("  Min absolute followers per country", 100_000) if audience_country else 0
    desired_interests  = prompt_list("  Desired interests (e.g. fashion, beauty)", [])

    print("\n── Music Filters ──")
    desired_genres   = prompt_list("  Genres (e.g. latin pop, reggaeton)", [])
    genre_match_mode = input("  Genre match mode (any / all) [any]: ").strip().lower() or "any"
    desired_moods    = prompt_list("  Moods (e.g. celebratory, sad)", [])
    mood_match_mode  = input("  Mood match mode (any / all) [any]: ").strip().lower() or "any"

    print("\n── Artist Filters ──")
    artist_country = prompt_list("  Artist countries (e.g. AR, CL, US)", [])
    print("  Artist reach:")
    for key, (label, mn, mx) in REACH_MAP.items():
        print(f"    {key}. {label} ({mn:,} – {mx:,} monthly listeners)")
    reach_choice        = input("  Select reach [3]: ").strip() or "3"
    _, min_pop, max_pop = REACH_MAP.get(reach_choice, REACH_MAP["3"])
    limit               = prompt_int("  Max artists to fetch per country", 200)

    # Fetch artists
    print("\n🔍 Fetching artists...\n")
    artists_raw = []
    for code in artist_country:
        params = {"min": min_pop, "max": max_pop, "code2": code, "limit": limit}
        res = Get(f"/api/artist/{metric}/list", params=params)
        if res and "obj" in res:
            batch = res["obj"]["data"]
            artists_raw.extend(batch)
            print(f"  {code}: {len(batch)} artists found")
        else:
            print(f"  {code}: ERROR retrieving artists")
    seen = set()
    artists_raw = [
        a for a in artists_raw
        if not (a["chartmetric_artist_id"] in seen or seen.add(a["chartmetric_artist_id"]))
    ]
    print(f"\n{len(artists_raw)} unique artists found.\n")

    filter_and_display(
        artists_raw, gender, min_gender_total, age, min_age_total,
        audience_country, min_audience_total, desired_interests,
        desired_genres, genre_match_mode, desired_moods, mood_match_mode
    )

In [9]:
# MODE B — FIND SIMILAR ARTISTS

def get_artist_id(name):
    res = Get("/api/search", params={"q": name, "type": "artists", "limit": 1})
    if not res or not res.get("obj") or not res["obj"].get("artists"):
        print(f"  ERROR: Could not find artist '{name}'")
        return None
    artist = res["obj"]["artists"][0]
    print(f"  Found: {artist['name']} (id: {artist['id']}, listeners: {(artist['sp_monthly_listeners'] or 0):,})")
    return artist["id"]

def get_similar_artists(artist_id, audience="high", mood="medium", genre="high", musicality="medium", limit=10):
    return Get(f"/api/artist/{artist_id}/similar-artists/by-configurations", params={
        "audience":   audience,
        "mood":       mood,
        "genre":      genre,
        "musicality": musicality,
        "limit":      limit,
    })

def run_mode_b(artist_names, audience_sim="medium", mood_sim="medium",
               genre_sim="medium", musicality_sim="medium",
               desired_reach="3", artist_country_filter=[],
               gender="female", min_gender_total=50_000,
               age=["18_24", "25_34"], min_age_total=10_000,
               audience_country=[], min_audience_total=100_000,
               desired_interests=[],
               desired_genres=[], genre_match_mode="any",
               desired_moods=[], mood_match_mode="any",
               limit=50):

    print("\n── Mode B: Find Similar Artists ──")

    # Resolve names to IDs
    print("\nResolving artist names...")
    featured_ids = []
    for name in artist_names:
        artist_id = get_artist_id(name)
        if artist_id:
            featured_ids.append(artist_id)
        time.sleep(0.3)

    if not featured_ids:
        print("ERROR: No artists could be resolved.")
        return

    _, min_pop, max_pop = REACH_MAP.get(desired_reach, REACH_MAP["3"])

    # Fetch similar artists
    print(f"\nFetching similar artists...\n")
    all_similar = {}

    for artist_id in featured_ids:
        res = get_similar_artists(artist_id, audience_sim, mood_sim, genre_sim, musicality_sim, limit)
        if not res or "obj" not in res:
            print(f"  ERROR: No similar artists found for id {artist_id}")
            continue

        for a in res["obj"]["data"]:
            aid = a.get("id")
            if aid in all_similar or aid in featured_ids:
                continue
            listeners = a.get("sp_monthly_listeners") or 0
            if not (min_pop <= listeners <= max_pop):
                continue
            if artist_country_filter:
                code = (a.get("code2") or "").upper()
                if code not in [c.upper() for c in artist_country_filter]:
                    continue
            all_similar[aid] = a

        time.sleep(0.3)

    artists_raw = sorted(all_similar.values(), key=lambda x: x.get("similarity", 0), reverse=True)
    print(f"{len(artists_raw)} similar artists found before demographic filter.\n")

    filter_and_display(
        artists_raw, gender, min_gender_total, age, min_age_total,
        audience_country, min_audience_total, desired_interests,
        desired_genres, genre_match_mode, desired_moods, mood_match_mode
    )

In [10]:
"""
LANDING SCREEN
"""

def run_syncfit():
    print("         🎵 SyncFit — Artist Discovery")
    print()
    print("How would you like to find artists?")
    print()
    print("  A — Search Based on Audience Demographics")
    print("      Search by audience demographics, genre, mood and popularity.")
    print("      You can use a reference movie's audience as your starting point.")
    print()
    print("  B — Find Similar Artists")
    print("      Find artists similar to those featured in a reference movie.")
    print()

    while True:
        choice = input("Select a mode (A / B): ").strip().upper()
        if choice in ["A", "B"]:
            break
        print("  ERROR: Invalid choice, please enter A or B.")

    print()
    if choice == "A":
        run_mode_a()
    elif choice == "B":
        run_mode_b()

In [11]:
#run_syncfit()

In [13]:
# TESTING WITH HARDCODED PROMPTS

# Test Mode A
gender             = "female"
min_gender_total   = 50_000
age                = []
min_age_total      = 0
audience_country   = ["PE", "MX", "CO"]
min_audience_total = 50_000
desired_interests  = []

desired_genres     = ["latin pop"]
genre_match_mode   = "any"
desired_moods      = []
mood_match_mode    = "any"

artist_country     = ["PE", "MX", "CO"]
_, min_pop, max_pop = REACH_MAP["3"]  # Mid-level
limit              = 200

# Fetch artists
artists_raw = []
for code in artist_country:
    params = {"min": min_pop, "max": max_pop, "code2": code, "limit": limit}
    res = Get(f"/api/artist/{metric}/list", params=params)
    if res and "obj" in res:
        batch = res["obj"]["data"]
        artists_raw.extend(batch)
        print(f"  {code}: {len(batch)} artists found")
seen = set()
artists_raw = [
    a for a in artists_raw
    if not (a["chartmetric_artist_id"] in seen or seen.add(a["chartmetric_artist_id"]))
]
print(f"\n{len(artists_raw)} unique artists found.\n")

filter_and_display(
    artists_raw, gender, min_gender_total, age, min_age_total,
    audience_country, min_audience_total, desired_interests,
    desired_genres, genre_match_mode, desired_moods, mood_match_mode
)

  PE: 33 artists found
  MX: 200 artists found
  CO: 200 artists found

433 unique artists found.

Genre/mood filter: 256 artists remaining.

Checking: Corazón Serrano...
  Followers: 651,475
  female: 277,616
  No match — countries below threshold: {'MX': 9018, 'CO': 0}
     total_followers: 651475
Checking: Grupo 5...
  Followers: 895,387
  female: 564,266
  ERROR: No country data for Grupo 5
Checking: Agua Marina...
  Followers: 12,942
  No match — female total 5,694 < min 50,000
     total_followers: 12942
Checking: La Bella Luz...
  ERROR: No stat data for La Bella Luz
Checking: Los 4...
  Followers: 224,400
  female: 143,468
  ERROR: No country data for Los 4
Checking: Armonía 10 de Walther Lozada...
  Followers: 263,126
  female: 107,620
  ERROR: No country data for Armonía 10 de Walther Lozada
Checking: La Única Tropical...
  Followers: 235,985
  female: 119,286
  No match — countries below threshold: {'MX': 0, 'CO': 0}
     total_followers: 235985
Checking: KID FLEX...
  Follo

KeyboardInterrupt: 

In [15]:
# Test Mode B
run_mode_b(
    artist_names          = ["Soda Stereo", "Luis Alberto Spinetta", "Rio"],
    audience_sim          = "high",
    mood_sim              = "medium",
    genre_sim             = "high",
    musicality_sim        = "high",
    desired_reach         = "2",
    artist_country_filter = ["AR", "PE"],
    gender                = "male",
    min_gender_total      = 10_000,
    age                   = ["18_24"],
    min_age_total         = 10_000,
    audience_country      = [],
    min_audience_total    = 0,
    desired_interests     = [],
    desired_genres        = [],
    genre_match_mode      = "any",
    desired_moods         = [],
    mood_match_mode       = "any",
    limit                 = 50
)


── Mode B: Find Similar Artists ──

Resolving artist names...
  Found: Soda Stereo (id: 915, listeners: 13,017,844)
  Found: Luis Alberto Spinetta (id: 91965, listeners: 1,652,752)
  Found: R.I.O. (id: 31067, listeners: 2,269,947)

Fetching similar artists...

17 similar artists found before demographic filter.

Genre/mood filter: 17 artists remaining.

Checking: Lisandro Aristimuño...
  Followers: 262,595
  male: 89,200
  18_24: 42,432
  ✅ MATCH
Checking: Ivan Noble...
  Followers: 203,057
  male: 50,673
  18_24: 27,067
  ✅ MATCH
Checking: Marilina Bertoldi...
  Followers: 244,056
  male: 95,814
  18_24: 60,659
  ✅ MATCH
Checking: 1915...
  Followers: 46,446
  male: 24,915
  18_24: 12,320
  ✅ MATCH
Checking: Los Gardelitos...
  Followers: 184,145
  male: 93,565
  18_24: 41,970
  ✅ MATCH
Checking: Ainda...
  Followers: 52,657
  male: 18,191
  No match — age buckets below threshold: {'18_24': 9808}
     total_followers: 52657
Checking: Isla de Caras...
  Followers: 27,917
  male: 12,61